# Proyecto 6: Dinámica de Robot 2-DOF

**Objetivo**: Modelado, simulación y visualización de un brazo robótico de 2 grados de libertad usando derivación simbólica (SymPy) e integración numérica (SciPy).

**Features demostradas**:
- Derivación simbólica de ecuaciones de Lagrange con SymPy
- `sympy-bridge.lp` para manipulación algebraica
- `defmath` para híbrido simbólico-numérico
- `scipy.integrate.solve_ivp` para integración numérica
- Controlador PD para seguimiento de trayectoria
- Visualización 2D de la animación del robot
- Exportación LaTeX de las ecuaciones de movimiento

## 1. Inicialización

In [ ]:
(load "lib/sympy-bridge.lp")
(load "lib/defmath.lp")
(load "lib/numpy-fast.lp")
(py:import "numpy")
(py:import "scipy.integrate")
(py:import "matplotlib.pyplot")
(py:import "matplotlib.animation")

## 2. Cinemática del Robot 2-DOF

El robot tiene dos eslabones de longitudes $l_1$ y $l_2$ con ángulos $\theta_1$ y $\theta_2$.

Posición del efector final:
- $x = l_1\sin(\theta_1) + l_2\sin(\theta_1 + \theta_2)$
- $y = -l_1\cos(\theta_1) - l_2\cos(\theta_1 + \theta_2)$

In [ ]:
;; Cinemática directa en Lisp
(defun forward-kinematics (theta1 theta2 l1 l2)
  (let ((x (+ (* l1 (sin theta1)) (* l2 (sin (+ theta1 theta2)))))
        (y (- (+ (* l1 (cos theta1)) (* l2 (cos (+ theta1 theta2)))))))
    (list x y)))

;; Verificar
(format t "Robot en reposo (θ₁=π/2, θ₂=0): ~A~%"
        (forward-kinematics (/ pi 2) 0 1 1))
(format t "Robot extendido (θ₁=0, θ₂=0): ~A~%"
        (forward-kinematics 0 0 1 1))

## 3. Derivación Simbólica de la Dinámica (Ecuaciones de Lagrange)

Derivamos las ecuaciones de movimiento usando mecánica Lagrangiana con SymPy.

In [ ]:
(py:run "
import sympy as sp

# Parámetros y variables
m1, m2, l1, l2, g = sp.symbols('m1 m2 l1 l2 g', positive=True)
t = sp.symbols('t')
theta1 = sp.Function('theta1')(t)
theta2 = sp.Function('theta2')(t)

# Velocidades angulares
dtheta1 = sp.diff(theta1, t)
dtheta2 = sp.diff(theta2, t)

# Posiciones de los centros de masa
x1 = l1*sp.sin(theta1)
y1 = -l1*sp.cos(theta1)
x2 = l1*sp.sin(theta1) + l2*sp.sin(theta1 + theta2)
y2 = -l1*sp.cos(theta1) - l2*sp.cos(theta1 + theta2)

# Energía cinética (asumiendo masas puntuales en los extremos)
T = 0.5*m1*(sp.diff(x1, t)**2 + sp.diff(y1, t)**2) + 0.5*m2*(sp.diff(x2, t)**2 + sp.diff(y2, t)**2)
T = sp.simplify(T)

# Energía potencial
V = m1*g*y1 + m2*g*y2

# Lagrangiano
L = T - V

print('Energía Cinética T:')
print(f'  {sp.latex(T)}')
print(f'\\nEnergía Potencial V:')
print(f'  {sp.latex(V)}')
")

### 3.1 Ecuaciones de Euler-Lagrange

$\frac{d}{dt}\frac{\partial L}{\partial \dot{\theta_i}} - \frac{\partial L}{\partial \theta_i} = \tau_i$

In [ ]:
(py:run "
# Ecuaciones de Lagrange
dL_dtheta1_dot = sp.diff(L, dtheta1)
dL_dtheta1 = sp.diff(L, theta1)
eq1 = sp.diff(dL_dtheta1_dot, t) - dL_dtheta1
eq1 = sp.simplify(eq1)

dL_dtheta2_dot = sp.diff(L, dtheta2)
dL_dtheta2 = sp.diff(L, theta2)
eq2 = sp.diff(dL_dtheta2_dot, t) - dL_dtheta2
eq2 = sp.simplify(eq2)

print('Ecuación 1 (τ₁):')
print(f'  {sp.latex(eq1)}')
print(f'\\nEcuación 2 (τ₂):')
print(f'  {sp.latex(eq2)}')
")

## 4. Matriz de Masa y Términos de Coriolis

Extraemos la matriz de inercia $M(q)$ y los términos de Coriolis/gravedad $C(q,\dot{q})\dot{q} + G(q)$.

In [ ]:
(py:run "
# Sustituir derivadas segundas por símbolos para extraer M(q)
ddtheta1, ddtheta2 = sp.symbols('ddtheta1 ddtheta2')
dt1, dt2 = sp.symbols('dt1 dt2')
th1, th2 = sp.symbols('th1 th2')

subs_dict = {
    sp.Derivative(theta1, (t, 2)): ddtheta1,
    sp.Derivative(theta2, (t, 2)): ddtheta2,
    sp.Derivative(theta1, t): dt1,
    sp.Derivative(theta2, t): dt2,
    theta1: th1,
    theta2: th2
}

eq1_s = sp.simplify(eq1.subs(subs_dict))
eq2_s = sp.simplify(eq2.subs(subs_dict))

# Extraer M(q) de la parte que multiplica a [ddtheta1, ddtheta2]
M11 = sp.simplify(sp.diff(eq1_s, ddtheta1))
M12 = sp.simplify(sp.diff(eq1_s, ddtheta2))
M21 = sp.simplify(sp.diff(eq2_s, ddtheta1))
M22 = sp.simplify(sp.diff(eq2_s, ddtheta2))

print('Matriz de Masa M(q):')
print(f'  M11 = {sp.latex(M11)}')
print(f'  M12 = {sp.latex(M12)}')
print(f'  M21 = {sp.latex(M21)}')
print(f'  M22 = {sp.latex(M22)}')

# Verificar simetría
print(f'\\nSimetría M12 == M21: {sp.simplify(M12 - M21) == 0}')
")

## 5. Simulación Numérica con solve_ivp

Creamos funciones numéricas con `lambdify` y simulamos.

In [ ]:
(py:run "
import numpy as np
from scipy.integrate import solve_ivp

# Parámetros del robot
params = {'m1': 2.0, 'm2': 1.0, 'l1': 1.0, 'l2': 0.8, 'g': 9.81}

# Lambdificar M(q) y términos restantes
M_func = sp.lambdify((th1, th2, m1, m2, l1, l2), 
                      sp.Matrix([[M11, M12], [M12, M22]]), 'numpy')

# Términos de Coriolis + Gravedad
C1 = sp.simplify(eq1_s - M11*ddtheta1 - M12*ddtheta2)
C2 = sp.simplify(eq2_s - M21*ddtheta1 - M22*ddtheta2)
C_func = sp.lambdify((th1, th2, dt1, dt2, m1, m2, l1, l2, g),
                      sp.Matrix([C1, C2]), 'numpy')

def robot_dynamics(t, state, m1, m2, l1, l2, g, tau):
    th1, th2, dt1, dt2 = state
    M = M_func(th1, th2, m1, m2, l1, l2)
    C_vec = C_func(th1, th2, dt1, dt2, m1, m2, l1, l2, g)
    tau_vec = np.array(tau(t)) if callable(tau) else np.array([0.0, 0.0])
    q_ddot = np.linalg.solve(M, tau_vec - C_vec).flatten()
    return [dt1, dt2, q_ddot[0], q_ddot[1]]

# Simulación: caída libre (sin torque)
sol = solve_ivp(
    robot_dynamics, [0, 5],
    [np.pi/2, 0.0, 0.0, 0.0],  # θ₁=90°, θ₂=0, velocidades 0
    args=(params['m1'], params['m2'], params['l1'], params['l2'], params['g'], [0.0, 0.0]),
    method='RK45', max_step=0.01, rtol=1e-6
)

print(f'Simulación completada: {len(sol.t)} pasos')
print(f'Tiempo final: {sol.t[-1]:.2f}s')
print(f'θ₁ final: {np.degrees(sol.y[0,-1]):.2f}°')
print(f'θ₂ final: {np.degrees(sol.y[1,-1]):.2f}°')

# Guardar trayectoria
t_hist = sol.t
th1_hist = sol.y[0]
th2_hist = sol.y[1]
")

## 6. Controlador PD para Seguimiento de Trayectoria

Implementamos un controlador PD que sigue una trayectoria deseada.

In [ ]:
(py:run "
# Controlador PD
Kp = np.array([100.0, 50.0])  # Ganancia proporcional
Kd = np.array([20.0, 10.0])   # Ganancia derivativa

# Trayectoria deseada: movimiento sinusoidal
def desired_trajectory(t):
    return np.array([np.pi/4 * np.sin(0.5 * t), np.pi/6 * np.sin(0.8 * t)])

def desired_velocity(t):
    return np.array([np.pi/4 * 0.5 * np.cos(0.5 * t), np.pi/6 * 0.8 * np.cos(0.8 * t)])

def pd_controller(t, state):
    th1, th2, dt1, dt2 = state
    q = np.array([th1, th2])
    dq = np.array([dt1, dt2])
    q_des = desired_trajectory(t)
    dq_des = desired_velocity(t)
    return Kp * (q_des - q) + Kd * (dq_des - dq)

# Simulación con control PD
sol_pd = solve_ivp(
    robot_dynamics, [0, 10],
    [0.0, 0.0, 0.0, 0.0],
    args=(params['m1'], params['m2'], params['l1'], params['l2'], params['g'], pd_controller),
    method='RK45', max_step=0.01, rtol=1e-6
)

print('Simulación PD completada')
th1_pd = sol_pd.y[0]
th2_pd = sol_pd.y[1]
t_pd = sol_pd.t
")

## 7. Visualización Completa

Gráficos de ángulos, velocidades y animación del robot.

In [ ]:
(py:run "
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle

fig = plt.figure(figsize=(16, 10))

# Subplot 1-2: Ángulos vs tiempo
ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(t_pd, np.degrees(th1_pd), 'b-', label='θ₁')
ax1.plot(t_pd, np.degrees(th2_pd), 'r-', label='θ₂')
ax1.set_xlabel('Tiempo (s)')
ax1.set_ylabel('Ángulo (°)')
ax1.set_title('Ángulos con Control PD')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Subplot 3: Trayectoria deseada vs real
ax2 = fig.add_subplot(2, 3, 2)
q1_des = [desired_trajectory(t)[0] for t in t_pd]
q2_des = [desired_trajectory(t)[1] for t in t_pd]
ax2.plot(t_pd, np.degrees(q1_des), 'b--', alpha=0.5, label='θ₁ deseado')
ax2.plot(t_pd, np.degrees(q2_des), 'r--', alpha=0.5, label='θ₂ deseado')
ax2.plot(t_pd, np.degrees(th1_pd), 'b-', label='θ₁ real')
ax2.plot(t_pd, np.degrees(th2_pd), 'r-', label='θ₂ real')
ax2.set_xlabel('Tiempo (s)')
ax2.set_title('Seguimiento de Trayectoria')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)

# Subplot 4: Error de seguimiento
ax3 = fig.add_subplot(2, 3, 3)
error1 = np.degrees(np.array(q1_des) - th1_pd)
error2 = np.degrees(np.array(q2_des) - th2_pd)
ax3.plot(t_pd, error1, 'b-', label='Error θ₁')
ax3.plot(t_pd, error2, 'r-', label='Error θ₂')
ax3.set_xlabel('Tiempo (s)')
ax3.set_ylabel('Error (°)')
ax3.set_title('Error de Seguimiento')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Subplot 5: Animación del robot (snapshots)
ax4 = fig.add_subplot(2, 1, 2)
ax4.set_xlim(-2.5, 2.5)
ax4.set_ylim(-2.5, 1.5)
ax4.set_aspect('equal')
ax4.grid(True, alpha=0.3)
ax4.set_title('Robot 2-DOF - Snapshots')

# Dibujar snapshots del robot
l1, l2 = params['l1'], params['l2']
colors = plt.cm.viridis(np.linspace(0, 1, 10))
for i, idx in enumerate(np.linspace(0, len(t_pd)-1, 10, dtype=int)):
    th1 = th1_pd[idx]
    th2 = th2_pd[idx]
    # Eslabón 1
    x1 = l1 * np.sin(th1)
    y1 = -l1 * np.cos(th1)
    # Eslabón 2
    x2 = x1 + l2 * np.sin(th1 + th2)
    y2 = y1 - l2 * np.cos(th1 + th2)
    ax4.plot([0, x1, x2], [0, y1, y2], '-o', color=colors[i], 
            linewidth=2, markersize=4, alpha=0.7)
    if i == 0:
        ax4.plot(x2, y2, 'go', markersize=8, label='Inicio')
    elif i == 9:
        ax4.plot(x2, y2, 'ro', markersize=8, label='Final')
ax4.plot(0, 0, 'ks', markersize=8, label='Base')
ax4.legend()

plt.tight_layout()
plt.savefig('robot-2dof.png', dpi=100)
plt.show()
")

## 8. Métricas de Rendimiento del Controlador

Calculamos el error RMS y el esfuerzo de control.

In [ ]:
;; Leer datos a Lisp y calcular métricas
(let* ((error1 (py:->lisp @error1))
       (error2 (py:->lisp @error2))
       (n (length error1)))
  (format t "=== Métricas de Control ===~%")
  (let* ((rms1 (sqrt (/ (reduce #'+ (mapcar (lambda (e) (* e e)) error1)) n)))
         (rms2 (sqrt (/ (reduce #'+ (mapcar (lambda (e) (* e e)) error2)) n)))
         (max1 (reduce #'max (mapcar #'abs error1)))
         (max2 (reduce #'max (mapcar #'abs error2))))
    (format t "Error RMS θ₁: ~,4F°~%" rms1)
    (format t "Error RMS θ₂: ~,4F°~%" rms2)
    (format t "Error Máx θ₁: ~,4F°~%" max1)
    (format t "Error Máx θ₂: ~,4F°~%" max2)))

## 9. Espacio de Trabajo del Robot

Visualizamos el espacio de trabajo alcanzable.

In [ ]:
(py:run "
# Muestrear espacio de trabajo
n_samples = 50
th1_range = np.linspace(0, 2*np.pi, n_samples)
th2_range = np.linspace(-np.pi, np.pi, n_samples)

workspace_x = []
workspace_y = []
for th1 in th1_range[::3]:
    for th2 in th2_range[::3]:
        x = l1 * np.sin(th1) + l2 * np.sin(th1 + th2)
        y = -(l1 * np.cos(th1) + l2 * np.cos(th1 + th2))
        workspace_x.append(x)
        workspace_y.append(y)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(workspace_x, workspace_y, s=1, alpha=0.5, c='blue')
ax.set_aspect('equal')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Espacio de Trabajo (l₁={l1}, l₂={l2})')

# Círculos de alcance máximo
from matplotlib.patches import Circle
r_max = l1 + l2
r_min = abs(l1 - l2)
ax.add_patch(Circle((0, 0), r_max, fill=False, color='red', linestyle='--', alpha=0.5))
ax.add_patch(Circle((0, 0), r_min, fill=False, color='green', linestyle='--', alpha=0.5))
ax.text(r_max+0.1, 0, f'r_max={r_max}', color='red')
ax.text(r_min+0.1, 0.2, f'r_min={r_min}', color='green')

ax.grid(True, alpha=0.3)
plt.savefig('workspace.png', dpi=100)
plt.show()
")

## 10. Ejercicios Propuestos

1. Agregar fricción viscosa en las articulaciones ($-b\dot{\theta}$)
2. Implementar un controlador de par calculado (computed torque)
3. Extender el modelo a 3-DOF (robot planar RRR)
4. Generar una trayectoria interpolada (via points) y seguirla con PD
5. Crear una animación MP4 completa del robot en movimiento
6. Exportar las ecuaciones de movimiento como código C para simulación en tiempo real